## Exploratory Data Analysis and Cleanup for Children's Books

In [2]:
# importing libraries
import pandas as pd
pd.set_option("display.max_columns", None)
import os
from dotenv import load_dotenv
import numpy as np
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ast
from collections import OrderedDict
import re

In [3]:
DetectorFactory.seed = 42

In [4]:
# paths
load_dotenv()
#children_path = os.getenv("CHILDREN_BOOKS")

True

In [5]:
#print(children_path)

In [6]:
comic_df = pd.read_json('../data/books/goodreads_books_history_biography.json', lines=True)

In [7]:
comic_df.shape

(302935, 29)

In [8]:
comic_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,1599150603,7,[],US,,"[{'count': '56', 'name': 'to-read'}, {'count':...",,false,4.13,B00DU10PUG,[],"Relates in vigorous prose the tale of Aeneas, ...",Paperback,https://www.goodreads.com/book/show/287141.The...,"[{'author_id': '3041852', 'role': ''}]",Yesterday's Classics,162,13,9781599150604,9,,2006,https://www.goodreads.com/book/show/287141.The...,https://s.gr-assets.com/assets/nophoto/book/11...,287141,46,278578,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls
1,184737297X,15,[169353],US,,"[{'count': '159', 'name': 'to-read'}, {'count'...",,false,3.93,B007YLTG5I,"[439108, 522621, 116770, 1275927, 6202655, 840...","London, 1196. At the command of Richard the Li...",Hardcover,https://www.goodreads.com/book/show/6066814-cr...,"[{'author_id': '37778', 'role': ''}]",Simon & Schuster UK,400,6,9781847372970,4,,2009,https://www.goodreads.com/book/show/6066814-cr...,https://images.gr-assets.com/books/1328724803m...,6066814,186,6243149,"Crowner Royal (Crowner John Mystery, #13)","Crowner Royal (Crowner John Mystery, #13)"
2,037583687X,615,[],US,,"[{'count': '4248', 'name': 'to-read'}, {'count...",,false,3.98,B0010SEMV4,"[614054, 272343, 824934, 581383, 93598, 638689...",It's 1953 and 11-year-old Penny dreams of a su...,Hardcover,https://www.goodreads.com/book/show/89377.Penn...,"[{'author_id': '137561', 'role': ''}]",Random House Books for Young Readers,288,25,9780375836879,7,,2006,https://www.goodreads.com/book/show/89377.Penn...,https://images.gr-assets.com/books/1320470906m...,89377,6949,86258,Penny from Heaven,Penny from Heaven
3,1400041694,44,[],US,en-US,"[{'count': '362', 'name': 'to-read'}, {'count'...",,false,3.75,B002OTKEP6,"[6516254, 9639119, 43147, 2511700, 11279431, 2...",A stunning and revealing examination of oil's ...,,https://www.goodreads.com/book/show/6158967-cr...,"[{'author_id': '31308', 'role': ''}]",,,,9781400041695,,,,https://www.goodreads.com/book/show/6158967-cr...,https://images.gr-assets.com/books/1320556043m...,6158967,338,6338156,Crude World: The Violent Twilight of Oil,Crude World: The Violent Twilight of Oil
4,8864116435,19,[],US,ita,"[{'count': '6463', 'name': 'to-read'}, {'count...",,true,4.28,,"[458275, 1289430, 6553843, 6760818, 776609, 39...",Stoner e il racconto della vita di un uomo tra...,ebook,https://www.goodreads.com/book/show/18628480-s...,"[{'author_id': '51229', 'role': ''}, {'author_...",Fazi,332,28,9788864116433,5,Le Strade #202,2012,https://www.goodreads.com/book/show/18628480-s...,https://images.gr-assets.com/books/1380976410m...,18628480,116,1559207,Stoner,Stoner


In [9]:
df = comic_df.copy()

| Column | Meaning |
|---|---|
| `isbn` | 10-digit ISBN for the book edition, if available. |
| `text_reviews_count` | Number of written/text reviews for this book edition. |
| `series` | List of Goodreads series IDs the book belongs to. Empty list means no series or unavailable. |
| `country_code` | Country code from Goodreads metadata, often `US`. |
| `language_code` | Language code for the edition, such as `eng`, `en-US`, etc. Can be blank. |
| `popular_shelves` | User-created Goodreads shelf tags for the book, with shelf name and count. |
| `asin` | Amazon Standard Identification Number, if available. |
| `is_ebook` | Whether the edition is an ebook, stored as `"true"` or `"false"`. |
| `average_rating` | Average Goodreads rating for the book, usually from 1 to 5. |
| `kindle_asin` | Kindle-specific Amazon ID, if available. |
| `similar_books` | List of Goodreads `book_id`s marked as similar books. |
| `description` | Book description or blurb text. Can be empty. |
| `format` | Book format, such as `Hardcover`, `Paperback`, or `Kindle Edition`. |
| `link` | Goodreads link for the book page. |
| `authors` | List of author objects, usually containing `author_id` and `role`. |
| `publisher` | Publisher name for this edition. |
| `num_pages` | Number of pages, if available. |
| `publication_day` | Publication day of the month, if available. |
| `isbn13` | 13-digit ISBN for the book edition, if available. |
| `publication_month` | Publication month, if available. |
| `edition_information` | Extra edition information, such as `Anniversary Edition` or `First Edition`. |
| `publication_year` | Publication year for this edition. |
| `url` | Goodreads URL for the book page. Usually similar to `link`. |
| `image_url` | URL for the book cover image. |
| `book_id` | Goodreads ID for this specific book edition. Useful for joining with reviews/interactions. |
| `ratings_count` | Number of ratings for this book edition. |
| `work_id` | Goodreads work ID. Groups multiple editions of the same underlying book. Useful for deduplication. |
| `title` | Full book title as shown on Goodreads, often including series information. |
| `title_without_series` | Book title with series information removed. Better for clean display/search. |

In [10]:
df.shape

(302935, 29)

In [11]:
df.columns

Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin',
       'similar_books', 'description', 'format', 'link', 'authors',
       'publisher', 'num_pages', 'publication_day', 'isbn13',
       'publication_month', 'edition_information', 'publication_year', 'url',
       'image_url', 'book_id', 'ratings_count', 'work_id', 'title',
       'title_without_series'],
      dtype='str')

In [12]:
df = df[['title', 'title_without_series', 'series', 'country_code', 'language_code',
       'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id', 'book_id',
       'text_reviews_count', 'average_rating', 'ratings_count', 'popular_shelves',
       'is_ebook', 'format', 'num_pages', 'publisher', 'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [13]:
df.shape

(302935, 26)

In [14]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls,[],US,,1599150603,,B00DU10PUG,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",false,Paperback,162,Yesterday's Classics,13,9,2006,,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[]
1,"Crowner Royal (Crowner John Mystery, #13)","Crowner Royal (Crowner John Mystery, #13)",[169353],US,,184737297X,,B007YLTG5I,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",false,Hardcover,400,Simon & Schuster UK,6,4,2009,,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840..."
2,Penny from Heaven,Penny from Heaven,[],US,,037583687X,,B0010SEMV4,9780375836879,86258,89377,615,3.98,6949,"[{'count': '4248', 'name': 'to-read'}, {'count...",false,Hardcover,288,Random House Books for Young Readers,25,7,2006,,It's 1953 and 11-year-old Penny dreams of a su...,"[{'author_id': '137561', 'role': ''}]","[614054, 272343, 824934, 581383, 93598, 638689..."
3,Crude World: The Violent Twilight of Oil,Crude World: The Violent Twilight of Oil,[],US,en-US,1400041694,,B002OTKEP6,9781400041695,6338156,6158967,44,3.75,338,"[{'count': '362', 'name': 'to-read'}, {'count'...",false,,,,,,,,A stunning and revealing examination of oil's ...,"[{'author_id': '31308', 'role': ''}]","[6516254, 9639119, 43147, 2511700, 11279431, 2..."
4,Stoner,Stoner,[],US,ita,8864116435,,,9788864116433,1559207,18628480,19,4.28,116,"[{'count': '6463', 'name': 'to-read'}, {'count...",true,ebook,332,Fazi,28,5,2012,Le Strade #202,Stoner e il racconto della vita di un uomo tra...,"[{'author_id': '51229', 'role': ''}, {'author_...","[458275, 1289430, 6553843, 6760818, 776609, 39..."


In [15]:
df['clean_title'] = df['title_without_series']

In [16]:
df.shape

(302935, 27)

### Detecting languages

In [17]:
def detect_lang(text):
    if pd.isna(text) or len(str(text).strip()) < 30:
        return np.nan

    try:
        lang = detect(str(text))
        return "eng" if lang == "en" else lang
    except:
        return np.nan

In [18]:
# normalize empty strings to NaN
df["language_code_final"] = df["language_code"].replace("", np.nan)

# create title + description text
df["language_text"] = (
    df["clean_title"].fillna("") + " " + df["description"].fillna("")
)

# detect only missing language codes
missing_lang_mask = df["language_code_final"].isna()

texts_to_detect = df.loc[missing_lang_mask, "language_text"]

detected_languages = [
    detect_lang(text)
    for text in tqdm(texts_to_detect, desc="Detecting languages")
]

df.loc[missing_lang_mask, "detected_language_code"] = detected_languages

# fill missing language code with detected language
df["language_code_final"] = df["language_code_final"].fillna(
    df["detected_language_code"]
)

# normalize English variants
df["language_code_final"] = (
    df["language_code_final"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "en": "eng",
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)

Detecting languages: 100%|██████████| 155335/155335 [06:28<00:00, 400.24it/s]


In [19]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,clean_title,language_code_final,language_text,detected_language_code
0,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls,[],US,,1599150603,,B00DU10PUG,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",false,Paperback,162,Yesterday's Classics,13,9,2006,,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[],The Aeneid for Boys and Girls,eng,The Aeneid for Boys and Girls Relates in vigor...,eng
1,"Crowner Royal (Crowner John Mystery, #13)","Crowner Royal (Crowner John Mystery, #13)",[169353],US,,184737297X,,B007YLTG5I,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",false,Hardcover,400,Simon & Schuster UK,6,4,2009,,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840...","Crowner Royal (Crowner John Mystery, #13)",eng,"Crowner Royal (Crowner John Mystery, #13) Lond...",eng
2,Penny from Heaven,Penny from Heaven,[],US,,037583687X,,B0010SEMV4,9780375836879,86258,89377,615,3.98,6949,"[{'count': '4248', 'name': 'to-read'}, {'count...",false,Hardcover,288,Random House Books for Young Readers,25,7,2006,,It's 1953 and 11-year-old Penny dreams of a su...,"[{'author_id': '137561', 'role': ''}]","[614054, 272343, 824934, 581383, 93598, 638689...",Penny from Heaven,eng,Penny from Heaven It's 1953 and 11-year-old Pe...,eng
3,Crude World: The Violent Twilight of Oil,Crude World: The Violent Twilight of Oil,[],US,en-US,1400041694,,B002OTKEP6,9781400041695,6338156,6158967,44,3.75,338,"[{'count': '362', 'name': 'to-read'}, {'count'...",false,,,,,,,,A stunning and revealing examination of oil's ...,"[{'author_id': '31308', 'role': ''}]","[6516254, 9639119, 43147, 2511700, 11279431, 2...",Crude World: The Violent Twilight of Oil,eng,Crude World: The Violent Twilight of Oil A stu...,nan
4,Stoner,Stoner,[],US,ita,8864116435,,,9788864116433,1559207,18628480,19,4.28,116,"[{'count': '6463', 'name': 'to-read'}, {'count...",true,ebook,332,Fazi,28,5,2012,Le Strade #202,Stoner e il racconto della vita di un uomo tra...,"[{'author_id': '51229', 'role': ''}, {'author_...","[458275, 1289430, 6553843, 6760818, 776609, 39...",Stoner,ita,Stoner Stoner e il racconto della vita di un u...,nan


In [20]:
df.columns

Index(['title', 'title_without_series', 'series', 'country_code',
       'language_code', 'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id',
       'book_id', 'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'is_ebook', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'clean_title', 'language_code_final', 'language_text',
       'detected_language_code'],
      dtype='str')

In [21]:
df.shape

(302935, 30)

In [22]:
df=df[['clean_title', 'series', 'language_code_final', 
       'isbn', 'isbn13', 'work_id', 'book_id', 
       'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [23]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,The Aeneid for Boys and Girls,[],eng,1599150603,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",Paperback,162,Yesterday's Classics,13,9,2006,,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[]
1,"Crowner Royal (Crowner John Mystery, #13)",[169353],eng,184737297X,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",Hardcover,400,Simon & Schuster UK,6,4,2009,,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840..."
2,Penny from Heaven,[],eng,037583687X,9780375836879,86258,89377,615,3.98,6949,"[{'count': '4248', 'name': 'to-read'}, {'count...",Hardcover,288,Random House Books for Young Readers,25,7,2006,,It's 1953 and 11-year-old Penny dreams of a su...,"[{'author_id': '137561', 'role': ''}]","[614054, 272343, 824934, 581383, 93598, 638689..."
3,Crude World: The Violent Twilight of Oil,[],eng,1400041694,9781400041695,6338156,6158967,44,3.75,338,"[{'count': '362', 'name': 'to-read'}, {'count'...",,,,,,,,A stunning and revealing examination of oil's ...,"[{'author_id': '31308', 'role': ''}]","[6516254, 9639119, 43147, 2511700, 11279431, 2..."
4,Stoner,[],ita,8864116435,9788864116433,1559207,18628480,19,4.28,116,"[{'count': '6463', 'name': 'to-read'}, {'count...",ebook,332,Fazi,28,5,2012,Le Strade #202,Stoner e il racconto della vita di un uomo tra...,"[{'author_id': '51229', 'role': ''}, {'author_...","[458275, 1289430, 6553843, 6760818, 776609, 39..."


In [24]:
df.shape

(302935, 21)

In [25]:
df = df[
    df["language_code_final"].astype("string").str.strip().str.lower() == "eng"
].copy()

In [26]:
df.shape

(244431, 21)

In [27]:
df['format'].value_counts().shape

(273,)

In [28]:
unique_formats = df["format"].dropna().unique().tolist()
print(unique_formats)

['Paperback', 'Hardcover', '', 'ebook', 'Mass Market Paperback', 'Kindle Edition', 'Audio CD', 'Audio', 'hardcover', 'Audible Audio', 'Audiobook', 'Paperback and eBook', 'Unknown Binding', 'Audio Cassette', 'Cloth', 'Trade Paperback', 'Overdrive Audiobook', 'Unbound', 'Spiral-bound', 'Large print', 'paperback', 'audiobook', 'paper', 'cloth', 'Large Print Paperback', 'Hardcover with dust jacket', 'Library Binding', 'Hardback', 'Leather Bound', 'Softcover', 'MP3 CD', 'Nook', 'Other Format', 'Audio CD - Narrated by Campbell Scott', 'Leatherbound', 'Paperback and ebook', 'Broche', 'hardback', 'Audible Audiobook', 'Board Book', 'comics', 'audiobook/video', 'Audio Book', 'Kindle', 'CD-ROM', 'hardbound', 'DVD', 'Hard Cover', 'School &amp; Library Binding', 'softcover', 'Hardcover, Slipcased', 'online fiction', 'Trade paperback', 'Standard Type', 'eARC', 'DVD-ROM', 'Box Set', 'mass market paperback', 'Quality Paperback', 'DVD and Guidebook', 'Perfect Paperback', 'Pamphlet', 'picture book', 'Pa

In [29]:
# 1. Define readable formats to keep
keep_formats = {
    # Standard print
    "Hardcover", "hardcover", "Hard Cover", "Hard cover", "hard cover",
    "Hardback", "hardback", "Hard Back", "Hard back", "Hardbound", "Hardbound",
    "hardbound", "HC", "Harcover",

    "Paperback", "paperback", "PAPERBACK", "paper back", "Paper back",
    "Papeback", "Paperbook", "PB", "Mass Market Paperback",
    "mass market paperback", "Mass market paperback", "Trade Paperback",
    "Trade paperback", "trade paperback", "Export Trade Paperback",
    "Quality Paperback", "Open Market Paperback", "Oversize Trade Paper",
    "B Format Paperback", "B Format paperback", "Large Paperback",
    "Large Print Paperback", "Paperback - Large Print",
    "Paperback (Large Print)", "Large print", "large print",
    "Large Print", "Hardcover (Large Print)", "Hardcover [LARGE PRINT]",
    "Softcover", "softcover", "Softcover/Paperback", "Softbound",
    "softback", "Soft cover", "Large Soft cover", "Perfect Paperback",
    "Print on Demand Paperback", "Print on Demand (Paperback)",
    "Field Guide Edition Paperback", "Deckle Edge Paperback",
    "Paperback, Sewn Binding", "Paperback, Paper Dust Jacket",
    "Paperback 22.4 x 15.2 x 1.6 cm", "paper", "Paper",

    # Print bindings
    "Cloth", "cloth", "Cloth Bound", "Decorated cloth",
    "2 volumes, bound in cloth", "Leather Bound", "Leatherbound",
    "Bonded Leather", "Leatherette Bound", "Leatherbound, hard-cover, 5 volumes",
    "Reinforced Cloth with Gilt Design and Coloured Top",
    "Library Binding", "Library Bound hardback", "School &amp; Library Binding",
    "school binding", "Textbook Binding", "Turtleback",
    "Spiral-bound", "Photocopy spiral bound", "Unbound",
    "Pamphlet", "pamphlet", "Booklet", "Chapbook",
    "Publisher's Binding", "Casebound", "Library", "Vinyl Bound",

    # Special print editions / sets
    "Hardcover with dust jacket", "Hardcover, Slipcased",
    "Hardcover in slipcase", "Hardcover in Slipcase",
    "Hardcover, Case bound", "Hardcover 1st edition",
    "Hardcover, 11&quot; x 11&quot;", "Over-Sized 12&quot;W x 16&quot;H Hardcover",
    "US Hardcover Edition", "Hardcover Boxset", "Hardcover Boxed Set",
    "Hardcover Picture Book", "Hard Paperback", "Pocket hardback",
    "Folio Hardback", "5 Volume Hardcover Set",
    "Box Set", "Paperback boxed set", "Paperback Boxed Set",
    "Book Club Omnibus", "Book", "book", "Advanced Reader Copy",
    "Uncorrected Proof", "paperback ARC", "ARC paperback", "ARC",
    "Novelty Book", "Board Book", "Board book", "picture book",
    "Comic Book", "comics", "Graphic Novel", "artist's book",
    "Field Notes Brand Book", "Bookprint",

    # Foreign-language labels that mean paperback/book
    "Broche", "Broschiert", "Poche", "Relie", "Livro de bolso",
    "Capa Mole", "pepabatsuku", "hadokaba",

    # Ebooks / readable digital text
    "ebook", "ePub", "epub", "eARC", "Kindle Edition", "Kindle",
    "kindle", "Nook", "Nook ebook", "Nookbook", "Nook Book",
    "NOOK Book", "Nook Edition", "Smashwords eBook for Nook, iBook, etc.",
    "MultiFormat eBook", "Electronically published book",
    "Audio eBook",  # has ebook, but review if you want strict no-audio
    "Pdf", "PDF", "pdf", ".pdf", "P.D.F.",
    "Softcover; PDF", "Paperback and downloadable PDF",
    "Paperback and eBook", "Paperback and ebook", "Paperback, ebook",
    "Paperback, e-book", "Paperback, Kindle eBook",
    "ebook/paperback", "ebook and print", "Print and ebook",
    "Paper and eBook", "Electronic, Paperback",
    "Paperback &amp; Hard cover", "Paperback &amp; Hardback",
    "Paperback/Hardback", "Hardcover, Paperback",

    # Online readable text
    "online fiction", "Online Fiction - Complete", "online",
    "Online Read", "free online", "free online text", "Blog",
    "Livejournal posting",
}

# 2. Define formats to drop
drop_formats = {
    # Missing / unknown
    "", "Unknown Binding", "Unknown", "unknown", "Other Format", "my books",

    # Audio
    "Audio", "audio", "Audio CD", "audio CD", "audio cd",
    "Audio CD (Unabridged)", "Audio CD [Unabridged]",
    "Audio CD - Narrated by Campbell Scott",
    "Audio CD - Narrated by Susan Lyons",
    "Audio CD - Narrated by Bill Bryson",
    "Audio CD - Narrated by Campbell Scott",
    "Audio CD - Narrated by Susan Lyons",
    "Audio CD - Narrated by Bill Bryson",
    "Audio CD - Abr", "Abriged Audio CD",
    "Unabridged Audio CD", "audio CD, unabridged",
    "Audio Book", "Audio book", "audio book",
    "Audiobook", "audiobook", "Audible Audio", "Audible Audiobook",
    "Audible Audio Edition", "Audible download", "Audible Download",
    "Audiobook - Audible Download", "Audiobook, Unabridged",
    "Unabridged Audiobook", "Abridged Audiobook",
    "Overdrive Audiobook", "eAudiobook",
    "Playaway", "Playaway Audiobook", "Playaway edition",
    "Preloaded Digital Audio Player", "Downloadable Audio File",
    "Downloadable audio file", "Audio Download",
    "Digital Audio", "Digital MP3 Audiobook",
    "MP3 CD", "MP3 Book", "MP3 Audio", "MP3", "MP3 audiobook",
    "mp3 Audiobook", "6 Audio CDs", "10 Audio CDs", "20 Audio CDs",
    "6 Audio Cassettes", "Four Audio Cassettes",
    "Forty-Two Audio Cassettes", "Audio Cassette", "audio cassette",
    "cassettes - unabridged", "(Audio CD)",
    "Book and 4 CD's", "audio course", "Audio  [With Earbuds]",

    # Video / discs / courses
    "CD-ROM", "CD", "cd", "DVD", "DVD-ROM", "DVD and Guidebook",
    "DVD and coursebook", "DVD, Course Guidebook", "Hardcover with DVD",
    "Video lectures",

    # Mixed media with audio/video
    "audiobook/video", "ebook + audiobook + videos",
    "Paperback + Audio CD", "Paperback w/ CD", "Paperback (with CD-ROM)",
    "Illustrated Book and CD Set", "Mixed Media",

    # Not really usable as book formats
    "824 pp", "revised edition", "Standard Type", "Slipcase", "oversized",
}


# 4. Clean the format column
df["format_clean"] = (df["format"].fillna("").astype(str).str.strip())

# 5. Assign each row to a format group
df["format_group"] = np.select(
    [
        df["format_clean"].isin(keep_formats),
        df["format_clean"].isin(drop_formats)
    ],
    ["text", "other"], default="review")

# 6. Check any unclassified formats
df["format_group"].value_counts()

format_group
text      175624
other      68783
review        24
Name: count, dtype: int64

In [30]:
df = df[df["format_group"] == "text"].copy()

In [31]:
df["format_group"].value_counts()

format_group
text    175624
Name: count, dtype: int64

In [32]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,format_clean,format_group
0,The Aeneid for Boys and Girls,[],eng,1599150603,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",Paperback,162,Yesterday's Classics,13,9,2006,,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[],Paperback,text
1,"Crowner Royal (Crowner John Mystery, #13)",[169353],eng,184737297X,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",Hardcover,400,Simon & Schuster UK,6,4,2009,,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840...",Hardcover,text
2,Penny from Heaven,[],eng,037583687X,9780375836879,86258,89377,615,3.98,6949,"[{'count': '4248', 'name': 'to-read'}, {'count...",Hardcover,288,Random House Books for Young Readers,25,7,2006,,It's 1953 and 11-year-old Penny dreams of a su...,"[{'author_id': '137561', 'role': ''}]","[614054, 272343, 824934, 581383, 93598, 638689...",Hardcover,text
7,Eighteenth-Century Europe: Tradition and Progr...,[414880],eng,0393929876,9780393929874,1409928,13707894,2,3.57,13,"[{'count': '53', 'name': 'to-read'}, {'count':...",Paperback,362,W. W. Norton & Company,9,1,2012,,The new second edition of the standard text on...,"[{'author_id': '8070', 'role': ''}, {'author_i...",[],Paperback,text
11,The Crimson Petal and the White,[],eng,015100692X,9780151006922,1210026,780911,94,3.87,681,"[{'count': '23959', 'name': 'to-read'}, {'coun...",Hardcover,838,Houghton Mifflin Harcourt,16,9,2002,Book of the Month Club Edition,"""Michel Faber leads us back to 1870s London, w...","[{'author_id': '16272', 'role': ''}]","[1153738, 1038223, 92286, 10387214, 44543, 727...",Hardcover,text


In [33]:
df.shape

(175624, 23)

In [34]:
df.columns

Index(['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'format_clean', 'format_group'],
      dtype='str')

In [35]:
df = df[['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format_clean', 'num_pages', 'publisher',
       'publication_year', 'description', 'authors', 'similar_books']]

In [36]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format_clean,num_pages,publisher,publication_year,description,authors,similar_books
0,The Aeneid for Boys and Girls,[],eng,1599150603,9781599150604,278578,287141,7,4.13,46,"[{'count': '56', 'name': 'to-read'}, {'count':...",Paperback,162,Yesterday's Classics,2006,"Relates in vigorous prose the tale of Aeneas, ...","[{'author_id': '3041852', 'role': ''}]",[]
1,"Crowner Royal (Crowner John Mystery, #13)",[169353],eng,184737297X,9781847372970,6243149,6066814,15,3.93,186,"[{'count': '159', 'name': 'to-read'}, {'count'...",Hardcover,400,Simon & Schuster UK,2009,"London, 1196. At the command of Richard the Li...","[{'author_id': '37778', 'role': ''}]","[439108, 522621, 116770, 1275927, 6202655, 840..."
2,Penny from Heaven,[],eng,037583687X,9780375836879,86258,89377,615,3.98,6949,"[{'count': '4248', 'name': 'to-read'}, {'count...",Hardcover,288,Random House Books for Young Readers,2006,It's 1953 and 11-year-old Penny dreams of a su...,"[{'author_id': '137561', 'role': ''}]","[614054, 272343, 824934, 581383, 93598, 638689..."
7,Eighteenth-Century Europe: Tradition and Progr...,[414880],eng,0393929876,9780393929874,1409928,13707894,2,3.57,13,"[{'count': '53', 'name': 'to-read'}, {'count':...",Paperback,362,W. W. Norton & Company,2012,The new second edition of the standard text on...,"[{'author_id': '8070', 'role': ''}, {'author_i...",[]
11,The Crimson Petal and the White,[],eng,015100692X,9780151006922,1210026,780911,94,3.87,681,"[{'count': '23959', 'name': 'to-read'}, {'coun...",Hardcover,838,Houghton Mifflin Harcourt,2002,"""Michel Faber leads us back to 1870s London, w...","[{'author_id': '16272', 'role': ''}]","[1153738, 1038223, 92286, 10387214, 44543, 727..."


In [37]:
def clean_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip() == "":
        return np.nan
    if isinstance(x, list) and len(x) == 0:
        return np.nan
    return x


def unique_list(series):
    seen = set()
    result = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        key = str(x)

        if key not in seen:
            seen.add(key)
            result.append(x)

    return result


def first_non_missing(series):
    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        return x

    return np.nan


def longest_text(series):
    texts = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, str) and x.strip() != "":
            texts.append(x.strip())

    if len(texts) == 0:
        return np.nan

    return max(texts, key=len)


def max_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.max()


def mean_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.mean()

In [38]:
numeric_cols = [
    "text_reviews_count",
    "average_rating",
    "ratings_count",
    "num_pages",
    "publication_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [39]:
agg_dict = {
    # Main identity fields
    "clean_title": first_non_missing,
    "authors": first_non_missing,
    "series": unique_list,
    "language_code_final": first_non_missing,

    # IDs for API calls / joins
    "isbn": unique_list,
    "isbn13": unique_list,
    "book_id": unique_list,

    # Edition-level metadata
    "publisher": unique_list,
    "publication_year": unique_list,

    # Book-level text/features
    "description": longest_text,
    "popular_shelves": first_non_missing,
    "similar_books": unique_list,

    # Numeric summary features
    "text_reviews_count": max_numeric,
    "ratings_count": max_numeric,
    "average_rating": mean_numeric,
    "num_pages": max_numeric,
}

# keep only columns that exist in df
agg_dict = {col: func for col, func in agg_dict.items() if col in df.columns}

books_work_df = (
    df.groupby("work_id", dropna=False)
    .agg(agg_dict)
    .reset_index()
)

In [40]:
rename_dict = {
    "series": "series_list",
    "isbn": "isbn_list",
    "isbn13": "isbn13_list",
    "book_id": "book_id_list",
    "publisher": "publisher_list",
    "publication_year": "publication_year_list",
    "similar_books": "similar_books_list",
}

books_work_df = books_work_df.rename(
    columns={k: v for k, v in rename_dict.items() if k in books_work_df.columns}
)

In [41]:
books_work_df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,104,Ines Of My Soul,"[{'author_id': '2238', 'role': ''}]",[],eng,"[0007257392, 0007241186, 9028422005, 000724116...","[9780007257393, 9780007241187, 9789028422001, ...","[907974, 837676, 7085172, 837664, 96440, 3300,...","[HarperPerennial, Harper Perennial, Wereldbibl...","[2007.0, 2008.0, 2006.0, 2014.0]","We schrijven het jaar 1580. Ines Suarez, een S...","[{'count': '2000', 'name': 'to-read'}, {'count...","[[1181998, 917491, 11210, 160431, 91659, 53927...",994,13754,3.900000,528.0
1,114,The Far Pavilions,"[{'author_id': '1040250', 'role': ''}]",[],eng,"[055312997X, 0140048332, 0241953022, 0553170082]","[9780553129977, 9780140048339, 9780241953020, ...","[10216, 11865756, 827093, 13444981, 19392999, ...","[Bantam Books, Penguin, Penguin Books, St. Mar...","[1979.0, 2011.0, 1978.0]",A magnificent romantic/historical/adventure no...,"[{'count': '1330', 'name': 'to-read'}, {'count...","[[578190, 673216, 146746, 369110, 112080, 1203...",36,238,4.200000,1191.0
2,172,Near a Thousand Tables: A History of Food,"[{'author_id': '3314630', 'role': ''}]",[],eng,[0743227409],[9780743227407],[16863],[Free Press],[2003.0],"In Near a Thousand Tables,acclaimed food histo...","[{'count': '861', 'name': 'to-read'}, {'count'...","[[1996737, 79959, 696667, 79958, 1528537, 7996...",33,432,3.680000,272.0
3,251,The Library of Greek Mythology,"[{'author_id': '15378', 'role': ''}, {'author_...",[],eng,"[0872910725, 0192839241, 0199536325]","[9780872910720, 9780192839244, 9780199536320]","[1389347, 27410, 4056720]","[Coronado Press (Lawrence, KS), Oxford Univers...","[1975.0, 1999.0, 2008.0]",The only work of its kind to survive from clas...,"[{'count': '162', 'name': 'to-read'}, {'count'...","[[423051, 141993, 1080876, 23520, 2455, 13337,...",23,842,3.986667,336.0
4,335,The Master,"[{'author_id': '1351903', 'role': ''}]",[],eng,"[0743250419, 0330485652, 0743250400, 0771085842]","[9780743250412, 9780330485654, 9780743250405, ...","[43691, 1052646, 1084126, 43697, 43692, 1457648]","[Scribner, Picador, Wheeler Publishing, Emblem...","[2005.0, 2004.0]","Like Michael Cunningham in The Hours,Colm Toib...","[{'count': '12697', 'name': 'to-read'}, {'coun...","[[3433301, 118192, 683006, 67719, 1001309, 925...",674,6442,3.830000,520.0


In [42]:
df = books_work_df.copy()

In [43]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,104,Ines Of My Soul,"[{'author_id': '2238', 'role': ''}]",[],eng,"[0007257392, 0007241186, 9028422005, 000724116...","[9780007257393, 9780007241187, 9789028422001, ...","[907974, 837676, 7085172, 837664, 96440, 3300,...","[HarperPerennial, Harper Perennial, Wereldbibl...","[2007.0, 2008.0, 2006.0, 2014.0]","We schrijven het jaar 1580. Ines Suarez, een S...","[{'count': '2000', 'name': 'to-read'}, {'count...","[[1181998, 917491, 11210, 160431, 91659, 53927...",994,13754,3.900000,528.0
1,114,The Far Pavilions,"[{'author_id': '1040250', 'role': ''}]",[],eng,"[055312997X, 0140048332, 0241953022, 0553170082]","[9780553129977, 9780140048339, 9780241953020, ...","[10216, 11865756, 827093, 13444981, 19392999, ...","[Bantam Books, Penguin, Penguin Books, St. Mar...","[1979.0, 2011.0, 1978.0]",A magnificent romantic/historical/adventure no...,"[{'count': '1330', 'name': 'to-read'}, {'count...","[[578190, 673216, 146746, 369110, 112080, 1203...",36,238,4.200000,1191.0
2,172,Near a Thousand Tables: A History of Food,"[{'author_id': '3314630', 'role': ''}]",[],eng,[0743227409],[9780743227407],[16863],[Free Press],[2003.0],"In Near a Thousand Tables,acclaimed food histo...","[{'count': '861', 'name': 'to-read'}, {'count'...","[[1996737, 79959, 696667, 79958, 1528537, 7996...",33,432,3.680000,272.0
3,251,The Library of Greek Mythology,"[{'author_id': '15378', 'role': ''}, {'author_...",[],eng,"[0872910725, 0192839241, 0199536325]","[9780872910720, 9780192839244, 9780199536320]","[1389347, 27410, 4056720]","[Coronado Press (Lawrence, KS), Oxford Univers...","[1975.0, 1999.0, 2008.0]",The only work of its kind to survive from clas...,"[{'count': '162', 'name': 'to-read'}, {'count'...","[[423051, 141993, 1080876, 23520, 2455, 13337,...",23,842,3.986667,336.0
4,335,The Master,"[{'author_id': '1351903', 'role': ''}]",[],eng,"[0743250419, 0330485652, 0743250400, 0771085842]","[9780743250412, 9780330485654, 9780743250405, ...","[43691, 1052646, 1084126, 43697, 43692, 1457648]","[Scribner, Picador, Wheeler Publishing, Emblem...","[2005.0, 2004.0]","Like Michael Cunningham in The Hours,Colm Toib...","[{'count': '12697', 'name': 'to-read'}, {'coun...","[[3433301, 118192, 683006, 67719, 1001309, 925...",674,6442,3.830000,520.0


In [44]:
df.shape

(122966, 17)

In [45]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description              6233
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                8272
dtype: int64

In [46]:
df.shape

(122966, 17)

In [47]:
df = df[df["description"].notna()].copy()

In [48]:
df.isna().sum()

work_id                     0
clean_title                 0
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description                 0
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                7186
dtype: int64

In [49]:
df.shape

(116733, 17)

In [50]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,104,Ines Of My Soul,"[{'author_id': '2238', 'role': ''}]",[],eng,"[0007257392, 0007241186, 9028422005, 000724116...","[9780007257393, 9780007241187, 9789028422001, ...","[907974, 837676, 7085172, 837664, 96440, 3300,...","[HarperPerennial, Harper Perennial, Wereldbibl...","[2007.0, 2008.0, 2006.0, 2014.0]","We schrijven het jaar 1580. Ines Suarez, een S...","[{'count': '2000', 'name': 'to-read'}, {'count...","[[1181998, 917491, 11210, 160431, 91659, 53927...",994,13754,3.900000,528.0
1,114,The Far Pavilions,"[{'author_id': '1040250', 'role': ''}]",[],eng,"[055312997X, 0140048332, 0241953022, 0553170082]","[9780553129977, 9780140048339, 9780241953020, ...","[10216, 11865756, 827093, 13444981, 19392999, ...","[Bantam Books, Penguin, Penguin Books, St. Mar...","[1979.0, 2011.0, 1978.0]",A magnificent romantic/historical/adventure no...,"[{'count': '1330', 'name': 'to-read'}, {'count...","[[578190, 673216, 146746, 369110, 112080, 1203...",36,238,4.200000,1191.0
2,172,Near a Thousand Tables: A History of Food,"[{'author_id': '3314630', 'role': ''}]",[],eng,[0743227409],[9780743227407],[16863],[Free Press],[2003.0],"In Near a Thousand Tables,acclaimed food histo...","[{'count': '861', 'name': 'to-read'}, {'count'...","[[1996737, 79959, 696667, 79958, 1528537, 7996...",33,432,3.680000,272.0
3,251,The Library of Greek Mythology,"[{'author_id': '15378', 'role': ''}, {'author_...",[],eng,"[0872910725, 0192839241, 0199536325]","[9780872910720, 9780192839244, 9780199536320]","[1389347, 27410, 4056720]","[Coronado Press (Lawrence, KS), Oxford Univers...","[1975.0, 1999.0, 2008.0]",The only work of its kind to survive from clas...,"[{'count': '162', 'name': 'to-read'}, {'count'...","[[423051, 141993, 1080876, 23520, 2455, 13337,...",23,842,3.986667,336.0
4,335,The Master,"[{'author_id': '1351903', 'role': ''}]",[],eng,"[0743250419, 0330485652, 0743250400, 0771085842]","[9780743250412, 9780330485654, 9780743250405, ...","[43691, 1052646, 1084126, 43697, 43692, 1457648]","[Scribner, Picador, Wheeler Publishing, Emblem...","[2005.0, 2004.0]","Like Michael Cunningham in The Hours,Colm Toib...","[{'count': '12697', 'name': 'to-read'}, {'coun...","[[3433301, 118192, 683006, 67719, 1001309, 925...",674,6442,3.830000,520.0


In [51]:
df = df.drop(columns=["language_code_final", "publication_year_list"])

In [52]:
df.shape

(116733, 15)

In [53]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,104,Ines Of My Soul,"[{'author_id': '2238', 'role': ''}]",[],"[0007257392, 0007241186, 9028422005, 000724116...","[9780007257393, 9780007241187, 9789028422001, ...","[907974, 837676, 7085172, 837664, 96440, 3300,...","[HarperPerennial, Harper Perennial, Wereldbibl...","We schrijven het jaar 1580. Ines Suarez, een S...","[{'count': '2000', 'name': 'to-read'}, {'count...","[[1181998, 917491, 11210, 160431, 91659, 53927...",994,13754,3.900000,528.0
1,114,The Far Pavilions,"[{'author_id': '1040250', 'role': ''}]",[],"[055312997X, 0140048332, 0241953022, 0553170082]","[9780553129977, 9780140048339, 9780241953020, ...","[10216, 11865756, 827093, 13444981, 19392999, ...","[Bantam Books, Penguin, Penguin Books, St. Mar...",A magnificent romantic/historical/adventure no...,"[{'count': '1330', 'name': 'to-read'}, {'count...","[[578190, 673216, 146746, 369110, 112080, 1203...",36,238,4.200000,1191.0
2,172,Near a Thousand Tables: A History of Food,"[{'author_id': '3314630', 'role': ''}]",[],[0743227409],[9780743227407],[16863],[Free Press],"In Near a Thousand Tables,acclaimed food histo...","[{'count': '861', 'name': 'to-read'}, {'count'...","[[1996737, 79959, 696667, 79958, 1528537, 7996...",33,432,3.680000,272.0
3,251,The Library of Greek Mythology,"[{'author_id': '15378', 'role': ''}, {'author_...",[],"[0872910725, 0192839241, 0199536325]","[9780872910720, 9780192839244, 9780199536320]","[1389347, 27410, 4056720]","[Coronado Press (Lawrence, KS), Oxford Univers...",The only work of its kind to survive from clas...,"[{'count': '162', 'name': 'to-read'}, {'count'...","[[423051, 141993, 1080876, 23520, 2455, 13337,...",23,842,3.986667,336.0
4,335,The Master,"[{'author_id': '1351903', 'role': ''}]",[],"[0743250419, 0330485652, 0743250400, 0771085842]","[9780743250412, 9780330485654, 9780743250405, ...","[43691, 1052646, 1084126, 43697, 43692, 1457648]","[Scribner, Picador, Wheeler Publishing, Emblem...","Like Michael Cunningham in The Hours,Colm Toib...","[{'count': '12697', 'name': 'to-read'}, {'coun...","[[3433301, 118192, 683006, 67719, 1001309, 925...",674,6442,3.830000,520.0


In [54]:
df["clean_title"] = (
    df["clean_title"]
    .astype(str)
    .str.replace(r"\s*\([^)]*#\d+[^)]*\)", "", regex=True)
    .str.strip()
)

In [55]:
def extract_author_ids(authors):
    if not isinstance(authors, list):
        return []

    return [
        str(a.get("author_id"))
        for a in authors
        if isinstance(a, dict) and a.get("author_id")
    ]

df["author_ids"] = df["authors"].apply(extract_author_ids)

In [56]:
def flatten_nested_list(x):
    if not isinstance(x, list):
        return []

    flat = []

    for item in x:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)

    # remove missing/empty values and dedupe
    result = []
    seen = set()

    for item in flat:
        if item is None:
            continue

        item_str = str(item).strip()

        if item_str == "" or item_str.lower() in {"nan", "none"}:
            continue

        if item_str not in seen:
            seen.add(item_str)
            result.append(item)

    return result

In [57]:
for col in ["series_list", "similar_books_list"]:
    if col in df.columns:
        df[col] = df[col].apply(flatten_nested_list)

In [58]:
numeric_cols = [
    "text_reviews_count",
    "ratings_count",
    "average_rating",
    "num_pages"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [59]:
df["num_pages_missing"] = df["num_pages"].isna().astype(int)

In [60]:
for i, shelves in enumerate(df["popular_shelves"].head(10), start=1):
    print(f"\nRow {i}:")
    print(shelves)


Row 1:
[{'count': '2000', 'name': 'to-read'}, {'count': '597', 'name': 'currently-reading'}, {'count': '392', 'name': 'historical-fiction'}, {'count': '261', 'name': 'fiction'}, {'count': '138', 'name': 'favorites'}, {'count': '67', 'name': 'chile'}, {'count': '62', 'name': 'historical'}, {'count': '58', 'name': 'latin-america'}, {'count': '54', 'name': 'isabel-allende'}, {'count': '53', 'name': 'books-i-own'}, {'count': '43', 'name': 'book-club'}, {'count': '39', 'name': 'owned'}, {'count': '35', 'name': 'spanish'}, {'count': '34', 'name': 'south-america'}, {'count': '33', 'name': 'history'}, {'count': '30', 'name': 'default'}, {'count': '30', 'name': 'novels'}, {'count': '29', 'name': 'romance'}, {'count': '26', 'name': 'latin-american'}, {'count': '23', 'name': 'owned-books'}, {'count': '23', 'name': 'literature'}, {'count': '22', 'name': 'magical-realism'}, {'count': '20', 'name': 'audio'}, {'count': '18', 'name': 'allende'}, {'count': '17', 'name': 'latin-american-literature'}, {

In [61]:
int_df = df[['popular_shelves']]

In [62]:
int_df.head()

,popular_shelves
0,"[{'count': '2000', 'name': 'to-read'}, {'count..."
1,"[{'count': '1330', 'name': 'to-read'}, {'count..."
2,"[{'count': '861', 'name': 'to-read'}, {'count'..."
3,"[{'count': '162', 'name': 'to-read'}, {'count'..."
4,"[{'count': '12697', 'name': 'to-read'}, {'coun..."


In [63]:
int_df.shape

(116733, 1)

In [64]:
int_df.to_csv('../data/intermediate/history_shelves.csv', index=False)

In [65]:
# TAG NORMALIZATION MAP ---> BUILT WITH CLAUDE
# instead of calling Claude or gpt's api or even using a model from HuggingFace, this seemed to be the fastest and the cheapest way
shelves = pd.read_csv('../data/intermediate/history_tag_mapping.csv')

In [66]:
shelves.head(20)

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,70219455,116027
1,currently-reading,status,drop,3510561,94158
2,historical-fiction,historical-fiction,keep,1529753,39810
3,non-fiction,nonfiction,keep,1137186,63767
4,history,history,keep,1133625,72657
5,fiction,fiction,keep,812647,36583
6,favorites,review,drop,739818,43861
7,historical,historical-fiction,keep,703622,50865
8,romance,romance,keep,645021,22927
9,historical-romance,historical-romance,keep,571311,18622


In [67]:
vocab_df = shelves.copy()

In [68]:
tag_map = dict(
    zip(
        vocab_df.loc[vocab_df["action"] == "keep", "raw_tag"],
        vocab_df.loc[vocab_df["action"] == "keep", "clean_tag"]
    )
)

In [69]:
vocab_df.head()

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,70219455,116027
1,currently-reading,status,drop,3510561,94158
2,historical-fiction,historical-fiction,keep,1529753,39810
3,non-fiction,nonfiction,keep,1137186,63767
4,history,history,keep,1133625,72657


In [70]:
def replace_shelf_tags(shelves):
    # if shelves got saved as a string, convert it back to list
    if isinstance(shelves, str):
        try:
            shelves = ast.literal_eval(shelves)
        except Exception:
            return []

    if not isinstance(shelves, list):
        return []

    cleaned_shelves = []

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        raw_tag = shelf.get("name")

        # only replace if action == keep and raw_tag exists in tag_map
        if raw_tag in tag_map:
            new_shelf = shelf.copy()
            new_shelf["name"] = tag_map[raw_tag]
            cleaned_shelves.append(new_shelf)

    return cleaned_shelves

In [71]:
df["popular_shelves_cleaned"] = df["popular_shelves"].apply(replace_shelf_tags)

In [72]:
df[["popular_shelves", "popular_shelves_cleaned"]].head()

,popular_shelves,popular_shelves_cleaned
0,"[{'count': '2000', 'name': 'to-read'}, {'count...","[{'count': '392', 'name': 'historical-fiction'..."
1,"[{'count': '1330', 'name': 'to-read'}, {'count...","[{'count': '634', 'name': 'historical-fiction'..."
2,"[{'count': '861', 'name': 'to-read'}, {'count'...","[{'count': '71', 'name': 'cooking'}, {'count':..."
3,"[{'count': '162', 'name': 'to-read'}, {'count'...","[{'count': '92', 'name': 'fairy-tales'}, {'cou..."
4,"[{'count': '12697', 'name': 'to-read'}, {'coun...","[{'count': '324', 'name': 'fiction'}, {'count'..."


In [73]:
print("Original popular_shelves:")
print(df.loc[df.index[0], "popular_shelves"])

print("\nCleaned popular_shelves:")
print(df.loc[df.index[0], "popular_shelves_cleaned"])

Original popular_shelves:
[{'count': '2000', 'name': 'to-read'}, {'count': '597', 'name': 'currently-reading'}, {'count': '392', 'name': 'historical-fiction'}, {'count': '261', 'name': 'fiction'}, {'count': '138', 'name': 'favorites'}, {'count': '67', 'name': 'chile'}, {'count': '62', 'name': 'historical'}, {'count': '58', 'name': 'latin-america'}, {'count': '54', 'name': 'isabel-allende'}, {'count': '53', 'name': 'books-i-own'}, {'count': '43', 'name': 'book-club'}, {'count': '39', 'name': 'owned'}, {'count': '35', 'name': 'spanish'}, {'count': '34', 'name': 'south-america'}, {'count': '33', 'name': 'history'}, {'count': '30', 'name': 'default'}, {'count': '30', 'name': 'novels'}, {'count': '29', 'name': 'romance'}, {'count': '26', 'name': 'latin-american'}, {'count': '23', 'name': 'owned-books'}, {'count': '23', 'name': 'literature'}, {'count': '22', 'name': 'magical-realism'}, {'count': '20', 'name': 'audio'}, {'count': '18', 'name': 'allende'}, {'count': '17', 'name': 'latin-americ

In [74]:
def merge_duplicate_shelf_tags(shelves):
    if not isinstance(shelves, list):
        return []

    merged = OrderedDict()

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        name = shelf.get("name")
        count = shelf.get("count", 0)

        if name is None or str(name).strip() == "":
            continue

        name = str(name).strip()

        try:
            count = int(count)
        except Exception:
            count = 0

        if name not in merged:
            merged[name] = count
        else:
            merged[name] += count

    return [
        {"count": str(count), "name": name}
        for name, count in merged.items()
    ]

In [75]:
df["popular_shelves_cleaned"] = df["popular_shelves_cleaned"].apply(
    merge_duplicate_shelf_tags
)

In [76]:
print(df.loc[df.index[1], "popular_shelves_cleaned"])

[{'count': '796', 'name': 'historical-fiction'}, {'count': '431', 'name': 'fiction'}, {'count': '236', 'name': 'india'}, {'count': '185', 'name': 'romance'}, {'count': '89', 'name': 'classics'}, {'count': '57', 'name': 'adventure'}, {'count': '48', 'name': 'asia'}, {'count': '33', 'name': 'historical-romance'}, {'count': '25', 'name': '19th-century'}, {'count': '23', 'name': 'history'}, {'count': '30', 'name': 'adult'}, {'count': '19', 'name': 'epic-fantasy'}, {'count': '13', 'name': 'fantasy'}, {'count': '12', 'name': 'multicultural'}, {'count': '18', 'name': 'british-history'}, {'count': '11', 'name': 'play'}, {'count': '10', 'name': 'war'}, {'count': '9', 'name': 'lgbt'}, {'count': '8', 'name': 'travel'}, {'count': '8', 'name': '20th-century'}, {'count': '7', 'name': 'image'}, {'count': '5', 'name': 'victorian'}, {'count': '4', 'name': 'middle-east'}]


In [77]:
for shelf in df.loc[df.index[10], "popular_shelves_cleaned"]:
    print(shelf)

{'count': '468', 'name': 'historical-fiction'}
{'count': '257', 'name': 'fiction'}
{'count': '62', 'name': 'adventure'}
{'count': '23', 'name': 'history'}
{'count': '39', 'name': 'war'}
{'count': '32', 'name': 'ocean'}
{'count': '10', 'name': 'classics'}
{'count': '8', 'name': 'thriller'}
{'count': '24', 'name': 'british-history'}
{'count': '8', 'name': 'regency'}
{'count': '5', 'name': '19th-century'}
{'count': '3', 'name': '20th-century'}


In [78]:
df.shape

(116733, 18)

In [79]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,author_ids,num_pages_missing,popular_shelves_cleaned
0,104,Ines Of My Soul,"[{'author_id': '2238', 'role': ''}]",[],"[0007257392, 0007241186, 9028422005, 000724116...","[9780007257393, 9780007241187, 9789028422001, ...","[907974, 837676, 7085172, 837664, 96440, 3300,...","[HarperPerennial, Harper Perennial, Wereldbibl...","We schrijven het jaar 1580. Ines Suarez, een S...","[{'count': '2000', 'name': 'to-read'}, {'count...","[1181998, 917491, 11210, 160431, 91659, 53927,...",994,13754,3.900000,528.0,[2238],0,"[{'count': '454', 'name': 'historical-fiction'..."
1,114,The Far Pavilions,"[{'author_id': '1040250', 'role': ''}]",[],"[055312997X, 0140048332, 0241953022, 0553170082]","[9780553129977, 9780140048339, 9780241953020, ...","[10216, 11865756, 827093, 13444981, 19392999, ...","[Bantam Books, Penguin, Penguin Books, St. Mar...",A magnificent romantic/historical/adventure no...,"[{'count': '1330', 'name': 'to-read'}, {'count...","[578190, 673216, 146746, 369110, 112080, 12038...",36,238,4.200000,1191.0,[1040250],0,"[{'count': '796', 'name': 'historical-fiction'..."
2,172,Near a Thousand Tables: A History of Food,"[{'author_id': '3314630', 'role': ''}]",[],[0743227409],[9780743227407],[16863],[Free Press],"In Near a Thousand Tables,acclaimed food histo...","[{'count': '861', 'name': 'to-read'}, {'count'...","[1996737, 79959, 696667, 79958, 1528537, 79962...",33,432,3.680000,272.0,[3314630],0,"[{'count': '123', 'name': 'cooking'}, {'count'..."
3,251,The Library of Greek Mythology,"[{'author_id': '15378', 'role': ''}, {'author_...",[],"[0872910725, 0192839241, 0199536325]","[9780872910720, 9780192839244, 9780199536320]","[1389347, 27410, 4056720]","[Coronado Press (Lawrence, KS), Oxford Univers...",The only work of its kind to survive from clas...,"[{'count': '162', 'name': 'to-read'}, {'count'...","[423051, 141993, 1080876, 23520, 2455, 13337, ...",23,842,3.986667,336.0,"[15378, 4000293, 5465793]",0,"[{'count': '110', 'name': 'fairy-tales'}, {'co..."
4,335,The Master,"[{'author_id': '1351903', 'role': ''}]",[],"[0743250419, 0330485652, 0743250400, 0771085842]","[9780743250412, 9780330485654, 9780743250405, ...","[43691, 1052646, 1084126, 43697, 43692, 1457648]","[Scribner, Picador, Wheeler Publishing, Emblem...","Like Michael Cunningham in The Hours,Colm Toib...","[{'count': '12697', 'name': 'to-read'}, {'coun...","[3433301, 118192, 683006, 67719, 1001309, 9257...",674,6442,3.830000,520.0,[1351903],0,"[{'count': '429', 'name': 'fiction'}, {'count'..."


In [80]:
df.columns

Index(['work_id', 'clean_title', 'authors', 'series_list', 'isbn_list',
       'isbn13_list', 'book_id_list', 'publisher_list', 'description',
       'popular_shelves', 'similar_books_list', 'text_reviews_count',
       'ratings_count', 'average_rating', 'num_pages', 'author_ids',
       'num_pages_missing', 'popular_shelves_cleaned'],
      dtype='str')

In [81]:
df = df[['work_id', 'clean_title', 'author_ids', 'series_list', 'isbn_list', 
         'isbn13_list', 'book_id_list', 'publisher_list', 'description', 
         'similar_books_list', 'text_reviews_count', 'ratings_count', 
         'average_rating', 'num_pages', 'num_pages_missing', 'popular_shelves_cleaned']]

In [82]:
df.head()

,work_id,clean_title,author_ids,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,num_pages_missing,popular_shelves_cleaned
0,104,Ines Of My Soul,[2238],[],"[0007257392, 0007241186, 9028422005, 000724116...","[9780007257393, 9780007241187, 9789028422001, ...","[907974, 837676, 7085172, 837664, 96440, 3300,...","[HarperPerennial, Harper Perennial, Wereldbibl...","We schrijven het jaar 1580. Ines Suarez, een S...","[1181998, 917491, 11210, 160431, 91659, 53927,...",994,13754,3.900000,528.0,0,"[{'count': '454', 'name': 'historical-fiction'..."
1,114,The Far Pavilions,[1040250],[],"[055312997X, 0140048332, 0241953022, 0553170082]","[9780553129977, 9780140048339, 9780241953020, ...","[10216, 11865756, 827093, 13444981, 19392999, ...","[Bantam Books, Penguin, Penguin Books, St. Mar...",A magnificent romantic/historical/adventure no...,"[578190, 673216, 146746, 369110, 112080, 12038...",36,238,4.200000,1191.0,0,"[{'count': '796', 'name': 'historical-fiction'..."
2,172,Near a Thousand Tables: A History of Food,[3314630],[],[0743227409],[9780743227407],[16863],[Free Press],"In Near a Thousand Tables,acclaimed food histo...","[1996737, 79959, 696667, 79958, 1528537, 79962...",33,432,3.680000,272.0,0,"[{'count': '123', 'name': 'cooking'}, {'count'..."
3,251,The Library of Greek Mythology,"[15378, 4000293, 5465793]",[],"[0872910725, 0192839241, 0199536325]","[9780872910720, 9780192839244, 9780199536320]","[1389347, 27410, 4056720]","[Coronado Press (Lawrence, KS), Oxford Univers...",The only work of its kind to survive from clas...,"[423051, 141993, 1080876, 23520, 2455, 13337, ...",23,842,3.986667,336.0,0,"[{'count': '110', 'name': 'fairy-tales'}, {'co..."
4,335,The Master,[1351903],[],"[0743250419, 0330485652, 0743250400, 0771085842]","[9780743250412, 9780330485654, 9780743250405, ...","[43691, 1052646, 1084126, 43697, 43692, 1457648]","[Scribner, Picador, Wheeler Publishing, Emblem...","Like Michael Cunningham in The Hours,Colm Toib...","[3433301, 118192, 683006, 67719, 1001309, 9257...",674,6442,3.830000,520.0,0,"[{'count': '429', 'name': 'fiction'}, {'count'..."


In [83]:
df.shape

(116733, 16)

In [85]:
df.to_csv('../data/intermediate/books-cleaned/history.csv', index=False)